In [1]:
!pip install giskard skl2onnx onnxruntime -q
from IPython.display import clear_output
clear_output()

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv('../data/processed/data_prepared.csv')

X = df.drop(columns=['sellingprice'])
y = df['sellingprice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

RandomForestRegressor(max_depth=20, n_jobs=-1, random_state=42)

## Giskard Report

In [3]:
import giskard

giskard_dataset = giskard.Dataset(
    df=X_test.assign(sellingprice=y_test.values),
    target='sellingprice',
    name='Vehicle Sales Dataset'
)

giskard_model = giskard.Model(
    model=model,
    model_type='regression',
    name='Random Forest - Vehicle Price',
    feature_names=list(X_train.columns)
)

scan_results = giskard.scan(giskard_model, giskard_dataset)

/opt/conda/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (7.4.3)/charset_normalizer (3.4.0) doesn't match a supported version!
  warnings.warn(


2026-05-17 14:08:24,550 pid:1832 MainThread giskard.datasets.base INFO     Your 'pandas.DataFrame' is successfully wrapped by Giskard's 'Dataset' wrapper class.
2026-05-17 14:08:29,391 pid:1832 MainThread giskard.models.automodel INFO     Your 'model' is successfully wrapped by Giskard's 'SKLearnModel' wrapper class.
2026-05-17 14:08:29,418 pid:1832 MainThread giskard.datasets.base INFO     Casting dataframe columns from {'year': 'float64', 'condition': 'float64', 'odometer': 'float64', 'body_convertible': 'bool', 'body_coupe': 'bool', 'body_crew cab': 'bool', 'body_crewmax cab': 'bool', 'body_double cab': 'bool', 'body_e-series van': 'bool', 'body_extended cab': 'bool', 'body_g coupe': 'bool', 'body_g sedan': 'bool', 'body_hatchback': 'bool', 'body_king cab': 'bool', 'body_minivan': 'bool', 'body_other': 'bool', 'body_quad cab': 'bool', 'body_regular cab': 'bool', 'body_sedan': 'bool', 'body_supercab': 'bool', 'body_supercrew': 'bool', 'body_suv': 'bool', 'body_unknown': 'bool', 'body

/opt/conda/lib/python3.12/site-packages/giskard/scanner/scanner.py:265: UserWarning: It looks like your dataset has a very large number of features (167), are you sure this is correct? The giskard.Dataset should be created from raw data *before* pre-processing (categorical encoding, vectorization, etc.). You can also limit the number of features to scan by setting the `features` argument. Check https://docs.giskard.ai/en/stable/guides/wrap_dataset/index.html for more details.
  warning(


2026-05-17 14:08:29,782 pid:1832 MainThread giskard.datasets.base INFO     Casting dataframe columns from {'year': 'float64', 'condition': 'float64', 'odometer': 'float64', 'body_convertible': 'bool', 'body_coupe': 'bool', 'body_crew cab': 'bool', 'body_crewmax cab': 'bool', 'body_double cab': 'bool', 'body_e-series van': 'bool', 'body_extended cab': 'bool', 'body_g coupe': 'bool', 'body_g sedan': 'bool', 'body_hatchback': 'bool', 'body_king cab': 'bool', 'body_minivan': 'bool', 'body_other': 'bool', 'body_quad cab': 'bool', 'body_regular cab': 'bool', 'body_sedan': 'bool', 'body_supercab': 'bool', 'body_supercrew': 'bool', 'body_suv': 'bool', 'body_unknown': 'bool', 'body_van': 'bool', 'body_wagon': 'bool', 'transmission_automatic': 'bool', 'transmission_manual': 'bool', 'state_ab': 'bool', 'state_al': 'bool', 'state_az': 'bool', 'state_ca': 'bool', 'state_co': 'bool', 'state_fl': 'bool', 'state_ga': 'bool', 'state_hi': 'bool', 'state_il': 'bool', 'state_in': 'bool', 'state_la': 'bool

In [5]:
import os
os.makedirs('../reports', exist_ok=True)
scan_results.to_html('../reports/giskard_report.html')


## Model Saving & Inference Speed Comparison

In [7]:
import pickle

with open('../reports/model_pickle.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('../reports/model_pickle.pkl', 'rb') as f:
    model_pkl = pickle.load(f)

In [8]:
%%timeit
model_pkl.predict(X_test)

86.9 ms ± 1.18 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [10]:
import joblib

joblib.dump(model, '../reports/model_joblib.joblib')
model_jb = joblib.load('../reports/model_joblib.joblib')

In [11]:
%%timeit
model_jb.predict(X_test)

86.5 ms ± 725 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [12]:
from skl2onnx import to_onnx
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as rt

initial_type = [('float_input', FloatTensorType([None, X_train.shape[1]]))]
onx = to_onnx(model, initial_types=initial_type)

with open('../reports/model.onnx', 'wb') as f:
    f.write(onx.SerializeToString())

sess = rt.InferenceSession('../reports/model.onnx')
X_test_f32 = X_test.astype(np.float32).values
print('ONNX model loaded!')

ONNX model loaded!


In [13]:
%%timeit
sess.run(None, {'float_input': X_test_f32})

101 ms ± 2.15 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Conclusion

Three model saving methods were compared using `%%timeit`:

| Method | Inference Time |
|--------|---------------|
| Pickle | 86.9 ms |
| Joblib | 86.5 ms |
| ONNX | 101 ms |

- **Pickle** — standard Python serialization, simple to use, good speed
- **Joblib** — slightly faster than Pickle, optimized for numpy arrays, recommended for sklearn models
- **ONNX** — cross-platform format, but slower here due to large number of features (167 columns)

**Joblib** is recommended for this model due to best inference speed and ease of use with sklearn.